# V3：LR/RF/XGBoost stacking

在共同外层折中比较静态 LR、RF、XGBoost 和 stacking。

第一层是三个固定参数的 Pipeline；没有在stacking子折里重复网格调参。第二层 LR 只用第一层在 stacking 内部验证折上的预测概率训练。每个基模型在该内部训练子集重新插补、标准化和LASSO选择，不在全体开发集上先生成特征，也不使用训练集拟合概率冒充折外预测。

外层验证患者从未参与上述过程。LR/RF/XGBoost 每个实验使用相同外层划分；第二层 C=1 预先固定，passthrough=False；stacking内部3折；LASSO内部5折/30λ。所有基模型参数见config.yaml，未依据当前6折结果重新挑选。

对比的新静态基线与扩展共享完整 Pipeline。它遵循 V3 的 λ.1se 原则，但 LASSO内部折也重新预处理，并采用固定基模型参数，不能要求与 V3 旧表逐位一致。

**balanced 简化计算版**：固定基模型参数、full保留5×10外层验证；LASSO内部5折/30λ，stacking内部3折。与上级严格版使用独立缓存；前6折旧预测不混入。当前没有运行新版本真实患者模型。

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# 从扩展目录或 notebooks 子目录定位项目，避免硬编码 Windows 盘符。
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "dn_extensions" / "settings.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("请从 dn_mortality_repro_V3_extensions 或其 notebooks 子目录启动。")
sys.path.insert(0, str(ROOT))
print("扩展目录：", ROOT)


In [ ]:
# pilot 是1×3外层小规模试跑；正式实验将 PROFILE 改为 "full"。
# 仅把 RUN_EXPERIMENT 改成 True 才会启动；full可能需要较长时间。
PROFILE = "pilot"
RUN_EXPERIMENT = False
EXPERIMENTS = ['static_lr', 'static_rf', 'static_xgb', 'static_stacking']
if RUN_EXPERIMENT:
    from dn_extensions.evaluation import run_evaluation
    from dn_extensions.settings import profile_options
    output, summary, comparisons = run_evaluation(profile_options(PROFILE), EXPERIMENTS)
    display(summary)
    display(comparisons)
else:
    print("尚未启动本Notebook中的模型训练。")


In [ ]:
# 优先查看正式 full 结果；没有结果时打印状态，不重新训练。
RESULT_PROFILE = "full"
latest = ROOT / "results" / RESULT_PROFILE / "latest.txt"
if latest.exists():
    folder = latest.parent / latest.read_text().strip()
    summary = pd.read_csv(folder / "summary.csv")
    display(summary[summary.experiment.isin(EXPERIMENTS)])
    print("详细记录：", folder / "report.md")
else:
    print("该模式尚无实验结果。pilot 结果需把 RESULT_PROFILE 改为 pilot 才显示。")


## 实验状态与结论（自动更新）

运行模式：`full`；方法：balanced 固定参数；结果目录：`results/full/27464ba98305edbc`。

LASSO 5折/30λ、stacking 3折；静态与扩展使用同一简化方法。严格版的部分折没有混入。

固定留出集未参与本实验。树模型扩展不需要 stacking 内部交叉拟合；原固定参数及训练折内预处理不变。

Holm 校正覆盖当前已有配对比较；后续增加实验时，校正 p 值可能变化。

```text

     experiment  n_outer_folds  AUROC_fold_mean  AUROC_fold_sd  AUPRC_fold_mean  Brier_fold_mean  calibration_slope_fold_mean  seconds_total  AUROC_pooled_mean  AUPRC_pooled_mean  Brier_pooled_mean  cal_intercept_pooled_mean  cal_slope_pooled_mean
      static_lr             50          0.79928        0.02648          0.60740          0.15092                      0.92073      502.87958            0.79885            0.59482            0.15091                    0.00385                0.89831
      static_rf             50          0.79798        0.02809          0.60618          0.15177                      1.19841       15.22184            0.79671            0.59435            0.15177                    0.00179                1.16211
static_stacking             50          0.80466        0.02657          0.61586          0.15048                      1.07477     1319.71249            0.80325            0.60386            0.15047                    0.00134                1.05015
     static_xgb             50          0.79504        0.02965          0.60572          0.15250                      0.94083        4.61352            0.79408            0.59662            0.15249                    0.00766                0.91690

```

配对差异：

```text

     experiment reference metric  mean_diff  sd_diff  se_corrected       lo      hi        t  df  p_value  k  r  p_holm_current_family
      static_rf static_lr  AUROC   -0.00130  0.01457       0.00528 -0.01190 0.00931 -0.24552  49  0.80708 10  5                    1.0
      static_rf static_lr  AUPRC   -0.00122  0.02900       0.01050 -0.02232 0.01989 -0.11569  49  0.90837 10  5                    1.0
      static_rf static_lr  Brier    0.00085  0.00469       0.00170 -0.00256 0.00427  0.50207  49  0.61787 10  5                    1.0
     static_xgb static_lr  AUROC   -0.00423  0.01409       0.00510 -0.01449 0.00602 -0.82968  49  0.41074 10  5                    1.0
     static_xgb static_lr  AUPRC   -0.00167  0.02894       0.01048 -0.02273 0.01938 -0.15972  49  0.87376 10  5                    1.0
     static_xgb static_lr  Brier    0.00158  0.00519       0.00188 -0.00220 0.00536  0.83926  49  0.40540 10  5                    1.0
static_stacking static_lr  AUROC    0.00539  0.00823       0.00298 -0.00060 0.01138  1.80933  49  0.07654 10  5                    1.0
static_stacking static_lr  AUPRC    0.00847  0.01519       0.00550 -0.00259 0.01952  1.53947  49  0.13012 10  5                    1.0
static_stacking static_lr  Brier   -0.00044  0.00289       0.00105 -0.00254 0.00166 -0.41929  49  0.67683 10  5                    1.0

```

- `static_rf` 相对 `static_lr`：ΔAUROC=-0.0013，校正95% CI [-0.0119, 0.0093]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

- `static_xgb` 相对 `static_lr`：ΔAUROC=-0.0042，校正95% CI [-0.0145, 0.0060]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

- `static_stacking` 相对 `static_lr`：ΔAUROC=0.0054，校正95% CI [-0.0006, 0.0114]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

## 2026-10-14 结果复核与解释

本次确实完成了四组模型各50个外层折，种子42–46。只使用V3开发集2045人、544例死亡，没有读取固定留出集标签。固定参数的 balanced 方法与上级严格版分开记录。

| 模型 | AUROC折均值 | AUPRC合并OOF均值 | Brier合并OOF均值 | 校准斜率合并OOF均值 |
|---|---:|---:|---:|---:|
| static_lr | 0.7993 | 0.5948 | 0.1509 | 0.898 |
| static_rf | 0.7980 | 0.5944 | 0.1518 | 1.162 |
| static_xgb | 0.7950 | 0.5966 | 0.1525 | 0.917 |
| static_stacking | 0.8047 | 0.6039 | 0.1505 | 1.050 |

**主要判断：stacking呈现小幅、方向较一致的改善，但证据不足以声称明确的性能提升。**

- AUROC折均值由 0.7993 到 0.8047。配对ΔAUROC=0.0054，Nadeau–Bengio校正95% CI [-0.0006, 0.0114]，p=0.0765；当前三个AUROC对比的Holm校正p=0.2296。区间跨0，不能写“显著提高”。
- 每次重复合并OOF的AUROC，stacking在5次中都高于LR；在50个外层折中，41折高于LR，9折低于LR。这说明方向有一致性，但这些结果共享患者，不能把50折当作50个独立实验做普通显著性检验。
- 合并OOF的平均AUROC分别为LR 0.7989、stacking 0.8032；与折均值属于不同汇总口径，报告需标明。
- AUPRC也略有改善，但按折配对Δ=0.0085，校正95% CI [-0.0026, 0.0195]，p=0.1301，仍不支持明确提升。
- stacking的校准斜率点估计更接近1（约1.05，对比LR约0.90），Brier仅略低。这是描述性结果，不等于已经证明校准改善具有统计学意义。
- DCA也没有显示stacking在整个阈值范围一致优于LR；在56个阈值中，平均净获益更高31个、更低25个。因此不能仅凭较高AUROC声称全面改善临床效用。

**耗时：** 所有模型记录的fit/predict计时相加约 30.7 分钟。Stacking每折平均 26.4 秒。该值是在与严格版同时运行时取得；RF和XGBoost复用了共同表示缓存，它们的单独计时不包含先前LR付出的表示学习成本。

**下一步：** 保持这套balanced设置不变，分别运行时间轨迹和burden/embedding消融，使用同一外层折和同一静态基线比较。不要因为这一轮尚未显著就按结果临时改超参数或反复查看固定留出集。

上述0.8047是重复外层CV平均AUROC，不是一次留出结果，也不能直接作为相对原论文0.780的确认性优势。


## 多重比较更新（2026-10-14）

新增两项消融后，当前每个指标共有6个配对对比。Stacking的ΔAUROC、置信区间和未作Holm校正的p值均未改变；Holm校正p更新为0.4592。此前复核记录中的数值对应当时3个对比。正式汇总请以最新paired_comparisons.csv为准。

# Stacking 最终开发集消融记录（2026-10-14）

已完成原定 5 次重复 × 10 折，共 50 折，seed 42–46。开发集 2,045 人、死亡 544 人。同一验证折作配对比较，未读取固定留出集标签，没有额外训练。这份完整记录取代前四次重复的暂定结论；中途快照保留用于追踪。

**结论：静态 stacking 相对当前最佳单模型 LR 有小幅数值提升，五次重复方向一致，但未达到统计显著。联合加入时间轨迹和既往诊断表示后，没有显示额外收益。**

| 模型 | AUROC（50折均值） | AUPRC（50折均值） | Brier（50折均值，低更好） |
|---|---:|---:|---:|
| 静态 LR | 0.799275 | 0.607397 | 0.150918 |
| 静态 RF | 0.797980 | 0.606182 | 0.151771 |
| 静态 XGBoost | 0.795041 | 0.605724 | 0.152496 |
| 静态 stacking | 0.804665 | 0.615863 | 0.150479 |
| 联合扩展 stacking | 0.804750 | 0.615715 | 0.150381 |

这些是先在每个外层验证折算指标，再对 50 折求平均。另一种“每次重复拼回完整开发集 OOF 概率后算指标，再对五次取平均”的 AUROC 分别为：静态 LR 0.798852、静态 stacking 0.803249、联合 stacking 0.803473。两种汇总方式不要混用。

## 静态 stacking 相对单模型

相对静态 LR，50 折平均 AUROC 增加 0.005389（约 0.54 个百分点），AUPRC 增加 0.008466，Brier 减少 0.000439。静态 stacking 的平均 AUROC 和 AUPRC 也高于当前固定参数的 RF、XGBoost；没有因此推断它优于所有调参后的单模型。

五次重复的 pooled OOF AUROC 和 AUPRC 均高于 LR；50 个折中 41 折的 AUROC 更高。重复使用同一开发集，这些并非五个独立外部验证样本。

Nadeau–Bengio 校正后的 AUROC 配对差值 95% 区间为 [−0.000596, +0.011375]，p=0.076535；对当前 7 个 AUROC 比较作 Holm 校正后 p=0.535748。AUPRC 差值区间 [−0.002585, +0.019518]，p=0.130124；Brier 差值区间 [−0.002541, +0.001663]，p=0.676835。因此可描述为“数值上有小幅提升，重复划分方向一致”，不能写成“显著优于 LR”。这里的差值区间不是 stacking 模型自身 AUROC 的置信区间。

## 联合扩展是否进一步提升 stacking

联合模型相对静态 stacking 的 50 折平均 AUROC 只增加 0.000086（约 0.009 个百分点），AUPRC 减少 0.000148，Brier 减少 0.000099，实际数值非常接近。

AUROC 配对差值 95% 区间为 [−0.007569, +0.007740]，p=0.982179，Holm p=1。五次重复的 pooled AUROC 三次上升、两次下降；50 折中 30 折上升。不能只凭赢的折数证明提升，也不能用未显著来证明严格等效。

| seed | 静态 stacking − LR AUROC | 联合 − 静态 stacking AUROC |
|---|---:|---:|
| 42 | +0.003559 | -0.001830 |
| 43 | +0.005730 | -0.000344 |
| 44 | +0.004561 | +0.000316 |
| 45 | +0.004749 | +0.001964 |
| 46 | +0.003383 | +0.001015 |

## 报告与后续实验的边界

当前可以将静态 stacking 作为候选模型，保留 LR 作为简单且有竞争力的对照。对联合模型，报告“当前扩展未带来明确额外增益”，无需为接近零的平均差值宣称改进。数值提升可能来自基模型预测互补，但本次指标比较没有直接验证提升机制。

本轮为固定基模型参数、简化内层计算的 balanced 版本，属于开发集探索性结果。不能与原文 0.780 直接比较并宣称超越原文，队列和评估方式不同。固定留出集若已看过，也不能当作全新未触碰测试集选择扩展。

独立的时间、诊断负担和 embedding 消融目前只评估了 LR。联合 stacking 的 LR、RF、XGBoost 均输入了联合扩展，但尚未单独报告 RF、XGBoost 的新增特征成绩。因此“联合没有收益”不能推广为“时间特征或 embedding 对每种模型都无效”。四个诊断负担指标与 ICD embedding 是在数值 LASSO 筛选后直接追加，未被 LASSO 排除；时间特征参与数值筛选。

会上可以说：

> 完整五次十折重复里，静态 stacking 的平均 AUROC 从 LR 的 0.799 提高到 0.805，五次重复均更高，但配对差值区间仍包含零，提升尚未得到统计学确认。加入时间轨迹和诊断表示后仍约 0.805，没有额外增益。目前静态 stacking 是一个有希望的候选，LR 仍是重要对照。
